In [1]:
import csv
from datetime import datetime

INPUT_FILE = "pp-complete.csv"      # from this huge existing file (more than 5GB, i had to create one smaller and work with that further)
OUTPUT_FILE = "pp-from-2013.csv"   
CUTOFF_DATE = datetime(2013, 1, 1)

DATE_COLUMN_INDEX = 2  # 3rd column: Date of Transfer

with open(INPUT_FILE, "r", encoding="utf-8", newline="") as infile, \
     open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as outfile:

    reader = csv.reader(infile)
    writer = csv.writer(outfile)

    for row in reader:
        if not row:
            continue

        try:
            transfer_date = datetime.strptime(
                row[DATE_COLUMN_INDEX], "%Y-%m-%d %H:%M"
            )
        except Exception:
            # skip broken lines safely
            continue

        if transfer_date >= CUTOFF_DATE:
            writer.writerow(row)

print("finished")



finished


In [2]:
INPUT_FILE = "pp-from-2013.csv"      
OUTPUT_FILE = "pp-from-2013-over-1m.csv"

PRICE_COLUMN_INDEX = 1  # 2nd column: Price
MIN_PRICE = 1_000_000

with open(INPUT_FILE, "r", encoding="utf-8", newline="") as infile, \
     open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as outfile:

    reader = csv.reader(infile)
    writer = csv.writer(outfile)

    for row in reader:
        if not row:
            continue

        try:
            price = int(row[PRICE_COLUMN_INDEX])
        except Exception:
            # skip malformed rows safely
            continue

        if price > MIN_PRICE:
            writer.writerow(row)

print("finished")


finished


In [2]:
import pandas as pd

INPUT_CSV = "pp-from-2013-over-1m.csv"
OUTPUT_CSV = "properties_over_1m_monthly.csv"

# Load data (no header assumed)
df = pd.read_csv(INPUT_CSV, header=None)

# Rename columns (based on your file structure)
df.columns = ["id", "price", "date"] + [f"col_{i}" for i in range(len(df.columns) - 3)]

# Convert types
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Drop bad rows
df = df.dropna(subset=["date", "price"])

# Filter >1M (keep just in case)
df = df[df["price"] > 1_000_000]

# Convert to monthly
df["month"] = df["date"].dt.to_period("M")

# Aggregate (count + average price per month)
monthly = df.groupby("month").agg(
    transactions=("price", "count"),
    avg_price=("price", "mean")
).reset_index()

# Convert period to string
monthly["month"] = monthly["month"].astype(str)

# Save output
monthly.to_csv(OUTPUT_CSV, index=False)

print("finished - monthly data saved")

finished - monthly data saved
